In [ ]:
import json

import matplotlib.pyplot as plt
import torch

In [ ]:
base_path = "reports/FIMImpPoint/SynthData_500k/8windows_25%overlap/MinMax_SavGol/"

predictions = torch.load(base_path + "predictions.pt")

In [ ]:
print(predictions["times_target"].shape)
print(predictions.keys())

# Visualizations

In [ ]:
# get 16 random samples
torch.manual_seed(42)
sample_ids = sorted(torch.randint(0, predictions["times_target"].size(0), (16,)))
sample_ids = [v.item() for v in sample_ids]
print(sample_ids)

In [ ]:
def plot_sample(line_plot_data: dict, sample_id: int, ax):
    colors = ["teal", "gold", "green", "red", "teal", "gold", "green"]  # noqa: F841

    obs_mask = line_plot_data["observation_mask"][sample_id].bool()
    obs_times = line_plot_data["observation_times"][sample_id][~obs_mask]
    obs_values = line_plot_data["observation_values"][sample_id][~obs_mask]

    ax.scatter(obs_times, obs_values, color="black", marker="x", label="observed")
    ax.plot(
        line_plot_data["times_target"][sample_id],
        line_plot_data["sample_paths_target"][sample_id],
        color="black",
        linestyle="--",
        label="target",
    )

    ax.plot(line_plot_data["times_target"][sample_id], line_plot_data["sample_path_prediction"][sample_id], color="blue", label="learnt")

    # axs[0].legend()
    ax.set_title("{}".format(sample_id))
    ax.set_xlabel("Time")

    # remove spines
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

In [ ]:
fig, axs = plt.subplots(4, 4, figsize=(20, 20))

for ax, sample_id in zip(axs.flatten(), sample_ids):
    plot_sample(predictions, sample_id, ax)
fig.suptitle("Sample Path prediction")
axs[0, 0].legend()
plt.tight_layout()
plt.savefig(base_path + "prediction.png")
plt.show()

# Metrics

In [ ]:
metrics: dict = json.load(open(base_path + "metrics.json", "r"))
print(json.dumps(metrics, indent=2))